# EABC-Qubit: Basis-Setup und erste Analyse

Dieses Notebook zeigt die grundlegende Verwendung des EABC-Qubit-Frameworks.

## Inhalt

1. Systemkonstruktion
2. Spektrumsberechnung
3. Level Spacing Distribution
4. Vergleich mit theoretischen Verteilungen

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

from src.hamiltonian import EABCHamiltonian
from src.spectral import spectral_unfolding
from src.level_spacing import compute_level_spacing, fit_level_statistics
from src.visualization import plot_level_statistics, plot_eigenspectrum

%matplotlib inline

ModuleNotFoundError: No module named 'seaborn'

## 1. Systemkonstruktion

Wir konstruieren ein EABC-Quantensystem mit:
- N = 1000 Gitterplätze
- α = 1.0 (Hopping-Stärke)
- β = 0.5 (Chirale Kopplung)
- γ = 1.5 (Primzahl-Defekt-Stärke)

In [ ]:
# System-Parameter
N = 1000
alpha = 1.0
beta = 0.5
gamma = 1.5

# Hamiltonian konstruieren
print("Konstruiere EABC-Hamiltonian...")
H = EABCHamiltonian(N, alpha, beta, gamma)

# System-Info
H.info()

## 2. Spektrumsberechnung

Wir berechnen 500 Eigenwerte im mittleren Energiebereich.

In [2]:
# Spektrum berechnen (mittlere Eigenwerte)
k = 500
print(f"\nBerechne {k} Eigenwerte...")
eigenvalues = H.compute_spectrum(k=k, which='SM')

print(f"\nErste 10 Eigenwerte:")
for i in range(10):
    print(f"  E_{i+1:3d} = {eigenvalues[i]:10.6f}")

print(f"\nSpektrum-Statistik:")
print(f"  Minimum:     {eigenvalues[0]:.6f}")
print(f"  Maximum:     {eigenvalues[-1]:.6f}")
print(f"  Bandwidth:   {eigenvalues[-1] - eigenvalues[0]:.6f}")
print(f"  Mean spacing: {np.mean(np.diff(eigenvalues)):.6f}")


Berechne 500 Eigenwerte...


NameError: name 'H' is not defined

## 3. Spektrales Unfolding

Normalisierung auf mittlere Dichte ρ̄ = 1.

In [4]:
# Spektrales Unfolding
print("Spektrales Unfolding...")
unfolded = spectral_unfolding(eigenvalues, method='polynomial')

print(f"\nEntfaltetes Spektrum:")
print(f"  Bereich: [{unfolded[0]:.3f}, {unfolded[-1]:.3f}]")
print(f"  Mittlere Dichte: {k / (unfolded[-1] - unfolded[0]):.3f} (sollte ≈ 1.0)")

# Visualisierung
plot_eigenspectrum(eigenvalues, unfolded, 
                   title=f"EABC-Qubit Spektrum (N={N}, γ={gamma})")

Spektrales Unfolding...


NameError: name 'spectral_unfolding' is not defined

## 4. Level Spacing Distribution

Die zentrale Größe zur Unterscheidung zwischen Poisson (integrabel) und Wigner-Dyson (chaotisch).

In [ ]:
# Level Spacings berechnen
print("Berechne Level Spacings...")
spacings = compute_level_spacing(unfolded)

print(f"\nSpacing-Statistik:")
print(f"  Anzahl: {len(spacings)}")
print(f"  Mittelwert: {np.mean(spacings):.6f} (sollte ≈ 1.0)")
print(f"  Std: {np.std(spacings):.6f}")
print(f"  Min: {np.min(spacings):.6f}")
print(f"  Max: {np.max(spacings):.6f}")

## 5. Fit gegen theoretische Verteilungen

In [ ]:
# Fit-Analyse
print("\nFit gegen theoretische Verteilungen...")
results = fit_level_statistics(spacings)

print("\nχ² Goodness-of-Fit:")
print(f"  Poisson: {results['poisson']:.3f}")
  print(f"  GOE:     {results['GOE']:.3f}")
print(f"  GUE:     {results['GUE']:.3f}")
print(f"  Brody:   {results['brody']:.3f} (q = {results['brody_q']:.3f})")

# Beste Fit
best_fit = min(results, key=lambda k: results[k] if k != 'brody_q' else np.inf)
print(f"\n→ Beste Übereinstimmung: {best_fit}")

# Interpretation
if best_fit == 'poisson':
    print("\n✓ ERGEBNIS: Poisson-Statistik")
    print("  → Primzahlen wirken wie unkorrelierte Störungen")
    print("  → Keine globale Quantenkohärenz")
elif best_fit in ['GOE', 'GUE']:
    print(f"\n✓ ERGEBNIS: {best_fit}-Statistik (Wigner-Dyson)")
    print("  → Level-Repulsion vorhanden")
    print("  → Quantenchaos / starke Korrelationen")
    print("  → Mögliche Verbindung zu Riemannschen Nullstellen!")
else:
    q = results['brody_q']
    print(f"\n✓ ERGEBNIS: Brody-Statistik (q = {q:.3f})")
    print(f"  → Intermediäres Regime zwischen Poisson und Wigner-Dyson")
    print(f"  → Partielles Chaos (q ∈ [0,1])")

## 6. Visualisierung

In [ ]:
# Plotte Level Statistics
plot_level_statistics(
    spacings,
    title=f"EABC-Qubit Level Statistics (N={N}, α={alpha}, β={beta}, γ={gamma})",
    show_theory=True
)

## Zusammenfassung

**Zentrale Frage**: Zeigt das EABC-Qubit-System Poisson- oder Wigner-Dyson-Statistik?

- **Poisson** → Primzahlen = unkorrelierte Störungen  
- **Wigner-Dyson (GUE/GOE)** → Kohärente Quantenstruktur, mögliche Verbindung zu Riemannschen Nullstellen

Nächste Schritte:
- Variation von γ (Defekt-Stärke)
- Größere Systeme (N = 10.000)
- Vergleich mit H₀ (ohne Primzahlen)